# One alternative in the ERD

Does a one-alternative specialization create a group or a relation line?

Run cells from top to bottom in a fresh Python kernel using the repository environment. Set the working directory to the repository root. This notebook is self-contained; do not run another declaration experiment in the same kernel. The matching `.py` file runs with `uv run python examples/step_21_relations/14_specialization_single_alternative.py`.

The record holds common information; the pressing classes hold details specific to one kind of pressing. A `Specialization` link goes from the common record to those details, and a `Generalization` link points back. The `Classifier` marker declares which data code belongs to which kind. The [relations chapter](../../docs/tutorials/step-21-relations.md#generalization-and-specialization) explains the complete preparation.


## Imports

These imports supply the annotation tools and AOA types used below.

In [1]:
from __future__ import annotations

from typing import Annotated, Literal

from pydantic import Field

from aoa.action_machine.domain import BaseEntity, Classifier, Generalization, Inverse, Rel, Specialization
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine


## MusicDomain

Group the music catalogue declarations. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [2]:
class MusicDomain(BaseDomain):
    """Group the music catalogue declarations."""

    name = "music"
    description = "A music catalogue"


## VinylRecordEntity

Describe the information shared by all pressings. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [3]:
@entity(description="Vinyl record", domain=MusicDomain)
class VinylRecordEntity(BaseEntity):
    """Describe the information shared by all pressings."""

    id: str = Field(description="Record identifier")
    title: str = Field(description="Album title")
    media: str = Field(description="Pressing code")
    pressing: Annotated[
        Specialization[FirstPressEntity],
        Classifier(field="media", codes=Literal["first"]),
        Inverse(field_name="record"),
    ] = Rel(description="Details of this pressing")


## FirstPressEntity

Describe the stamper used for a first pressing. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [4]:
@entity(description="First pressing", domain=MusicDomain)
class FirstPressEntity(BaseEntity):
    """Describe the stamper used for a first pressing."""

    id: str = Field(description="Pressing identifier")
    stamper: str = Field(description="Stamper code")
    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["first"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record described by this pressing")


## Resolve the declared types

Pydantic can now resolve the references between classes. This does not create record objects or load data.

In [5]:
VinylRecordEntity.model_rebuild()


## Resolve the declared types

Pydantic can now resolve the references between classes. This does not create record objects or load data.

In [6]:
FirstPressEntity.model_rebuild()


## The experiment

Does a one-alternative specialization create a group or a relation line?

In [7]:
import json

from aoa.maxitor.model.diagrams.actions.list_entities_action import ListEntitiesAction
from aoa.maxitor.model.diagrams.resources.duckdb_graph_resource import DuckDBGraphResource

machine = ActionProductMachine(loggers=[])
store = DuckDBGraphResource.build_from_json(json.loads(machine.graph_coordinator.to_json()))
domain_id = next(node.node_id for node in machine.graph_coordinator.get_all_nodes() if node.label == "MusicDomain")
diagram = ListEntitiesAction._slice_payload(store, domain_id, include_neighbors=False)
print("Groups:", len(diagram["groups"]))
print("Relation lines:", len(diagram["relations"]))


Groups: 0
Relation lines: 0


## Why this result occurs

The declaration builds with one alternative. The current ERD query emits neither a group nor a replacement ordinary relation line for this specialization. Its field remains in the entity table. Do not infer that a missing diagram line means the declaration has no relationship.